# GradMem Idea 3 on Kaggle (with combined_n and fused)

A copy of `gradmem_idea3_kaggle_n.ipynb` with two more run types, `fused` (from an RMT parent) and `fused_reference` (from scratch). They train `run_gradmem_fused_on_mqar.py`, GradMem whose write step also takes an RMT-style memory from the same forward pass (`fused_mode`, `fused_norm`).

A copy of `gradmem_idea3_kaggle.ipynb` with one more run type, `combined_n`. It trains `run_enc_gradmem_n_on_mqar.py`, the combined model with the write-step rule (`inner_step_scale`, N1) and the random start (`start_mode='random'`, C4). Everything else is the same.

This notebook trains a queue of runs, two at a time, one on each T4 of the "GPU T4 x2" accelerator. When a run ends, the next run in the queue takes its GPU, so no GPU sits idle. Everything is saved in the notebook's output.

How runs survive between sessions
- Kaggle keeps only the files in `/kaggle/working`, and only when a version is saved. Here everything that must survive is in `/kaggle/working/gradmem_idea3` (run folders with checkpoints, logs, `results.csv`, `runs.md`).
- Training itself runs in `/tmp`, and the helper mirrors complete checkpoints and logs into `/kaggle/working` every 10 minutes and once more at the end.
- In the next session, attach the previous version's output (and the dataset with runs from Drive) with **Add Input**. Cell 5 finds every run folder in the inputs, takes the newest copy of each, and the runs continue from their last checkpoint.

How to run a session
1. Settings panel. Accelerator **GPU T4 x2**, Internet **on**.
2. Set the runs in cell 6.
3. **Save Version**, then **Save & Run All (Commit)**. The notebook runs in the background, and the browser can be closed.
4. When it finishes, open the version's **Output** tab. The results are in `gradmem_idea3/results/`.

The session stops the runs by itself after `TIME_LIMIT_HOURS` (default 11), before Kaggle's 12-hour limit, so the output is always saved. **Set it below your remaining weekly GPU quota** (shown in your Kaggle settings), because a session cut by the quota may not save its output.

Before the first long session, check the setup cheaply. Start an interactive session with the GPU, set `DRY_RUN = True` in cell 6 and run all. Cells 1 to 6 then build every configuration and stop before training. Fix anything they refuse, stop the session, set `DRY_RUN = False` and save a version.

## 1. Settings

In [ ]:
import time
SESSION_START = time.time()

FORK_URL = 'https://github.com/alex-semin-ai/gradmem.git'  # your fork
BRANCH = 'my-idea3'                                        # your branch in the fork
INPUT_ROOT = '/kaggle/input'                               # attached datasets and earlier outputs
PROJ = '/kaggle/working/gradmem_idea3'                     # saved as the version's output
REPO = '/tmp/gradmem'                                      # code, not saved
LOCAL_RUNS = '/tmp/runs'                                   # fast training folders, not saved
LOGS = '/tmp/logs'                                         # console logs (the helper copies them into PROJ)
HELPER = '/tmp/run_helper.py'
SYNC_MINUTES = 10                                          # how often runs are mirrored into PROJ

## 2. Get the code

Needs Internet on in the settings panel. For a private fork, add a Kaggle secret named `GITHUB_TOKEN` (Add-ons, Secrets). Never paste a token into a cell.

In [ ]:
import os, subprocess

def clone_url():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        token = None
    if token:
        return FORK_URL.replace('https://', f'https://x-access-token:{token}@')
    return FORK_URL

if not os.path.isdir(f'{REPO}/.git'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, clone_url(), REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q', 'origin', BRANCH], check=True)

COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True, check=True).stdout.strip()
print('Code version:', BRANCH, COMMIT)

## 3. Install packages and check the GPUs

transformers is pinned to 4.51.3, as on Colab. Kaggle's own torch is kept. Version warnings from other preinstalled packages can be ignored.

In [ ]:
!pip install -q transformers==4.51.3

import os
os.environ['USE_TF'] = '0'     # transformers skips the preinstalled TensorFlow (training uses PyTorch)
os.environ['USE_TORCH'] = '1'

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
N_GPUS = torch.cuda.device_count()
assert N_GPUS > 0, 'No GPU. Set the accelerator to GPU T4 x2 in the settings panel.'
print('GPUs:', N_GPUS)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 4. Write the run helper

The same helper as in the Colab notebook. Here its "Drive" folder is the run folder in `/kaggle/working/gradmem_idea3/runs`. It resumes from the newest complete checkpoint, mirrors checkpoints and logs, keeps `summary.json`, `results.csv` and `runs.md`, refuses runs that break the setup rules (a `gradmem` run must start from an RMT checkpoint trained with the reconstruction loss), and stops a run whose starting checkpoint leaves weights unloaded.

In [ ]:
%%writefile /tmp/run_helper.py
"""Run a repo training script with resume and Drive mirroring.

Usage
    python colab_run.py --repo R --script S --local_dir L --drive_dir D --log F --sync_minutes M -- <script args>
The script args must contain --exp_path equal to --local_dir.
"""
import argparse, csv, json, math, os, re, runpy, shutil, signal, sys, threading, time
from pathlib import Path

REQUIRED = ('trainer_state.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth')
MODEL_FILES = ('model.safetensors', 'pytorch_model.bin')


def step_of(p):
    m = re.fullmatch(r'checkpoint-(\d+)', p.name)
    return int(m.group(1)) if m else -1


def is_complete(p):
    ok = all((p / f).is_file() and (p / f).stat().st_size > 0 for f in REQUIRED)
    return ok and any((p / f).is_file() for f in MODEL_FILES)


def is_stable(p, seconds=3):
    """True when the Trainer has finished writing the checkpoint.

    The Trainer writes trainer_state.json last (after weights, optimizer, scheduler and RNG state),
    so a readable trainer_state.json plus a few quiet seconds means the folder is complete."""
    try:
        json.load(open(p / 'trainer_state.json'))
    except (OSError, ValueError):
        return False
    newest = max((f.stat().st_mtime for f in p.rglob('*') if f.is_file()), default=0)
    return time.time() - newest > seconds


def complete_ckpts(d, stable=False):
    if not d.is_dir():
        return []
    found = [p for p in d.iterdir() if p.is_dir() and step_of(p) >= 0 and is_complete(p)]
    if stable:
        found = [p for p in found if is_stable(p)]
    return sorted(found, key=step_of)


def copy_dir_atomic(src, dst):
    tmp = dst.with_name(dst.name + '.tmp')
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp)
    shutil.rmtree(dst, ignore_errors=True)
    os.rename(tmp, dst)


def copy_if_newer(src, dst):
    if dst.exists():
        s, d = src.stat(), dst.stat()
        if s.st_size == d.st_size and s.st_mtime <= d.st_mtime:
            return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def restore(local, drive):
    """Copy checkpoints and files from Drive when the local run folder has no checkpoint."""
    if complete_ckpts(local) or not drive.is_dir():
        return
    for c in complete_ckpts(drive / 'checkpoints'):
        copy_dir_atomic(c, local / c.name)
    files = drive / 'files'
    if files.is_dir():
        for p in files.rglob('*'):
            if p.is_file():
                copy_if_newer(p, local / p.relative_to(files))
    n = len(complete_ckpts(local))
    if n:
        print(f'[run] restored {n} checkpoint(s) from Drive', flush=True)


def mirror(local, drive, log, final=False):
    """Mirror complete checkpoints first, then all other files of the run, to Drive."""
    (drive / 'checkpoints').mkdir(parents=True, exist_ok=True)
    # 1. checkpoints. During training only checkpoints the Trainer finished writing are copied.
    local_ck = complete_ckpts(local, stable=not final)
    names = {c.name for c in local_ck}
    for c in local_ck:
        if not (drive / 'checkpoints' / c.name).is_dir():
            copy_dir_atomic(c, drive / 'checkpoints' / c.name)
    # remove a Drive checkpoint only when it is gone locally AND a newer one is already on Drive,
    # so Drive always keeps the newest checkpoint it has seen
    if local_ck:
        newest = max(step_of(c) for c in local_ck)
        for c in (drive / 'checkpoints').iterdir():
            if c.is_dir() and 0 <= step_of(c) < newest and c.name not in names:
                shutil.rmtree(c, ignore_errors=True)
    # 2. small files (TensorBoard logs, config, console log), one by one so one failure blocks nothing
    files = [(p, drive / 'files' / p.relative_to(local)) for p in local.rglob('*')
             if p.is_file() and not p.relative_to(local).parts[0].startswith('checkpoint-')]
    if log and Path(log).is_file():
        files.append((Path(log), drive / 'logs' / Path(log).name))
    for src, dst in files:
        try:
            copy_if_newer(src, dst)
        except OSError as e:
            print(f'[run] could not copy {src.name}: {e}', flush=True)


# ---------- run log, summary.json, runs.md and results.csv ----------

def now():
    return time.strftime('%Y-%m-%d %H:%M')


def replace_retry(tmp, path, tries=50):
    # os.replace, retried briefly. On Windows a file another process has open cannot be replaced
    for _ in range(tries - 1):
        try:
            return os.replace(tmp, path)
        except PermissionError:
            time.sleep(0.1)
    os.replace(tmp, path)


def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + f'.{os.getpid()}.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf8')
    replace_retry(tmp, path)


def append_md(path, text):
    if not path:
        return
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, 'a', encoding='utf8') as f:
        f.write(text.rstrip('\n') + '\n\n')


NOT_METRICS = ('loss', 'runtime', 'per_second', 'epoch')


def eval_summary(log_history, best_metric, metric_name=None):
    """Accuracy key, best accuracy and the first eval step that reached 99%."""
    evals = [h for h in log_history if any(k.startswith('eval_') for k in h)]
    keys = sorted({k for h in evals for k in h
                   if k.startswith('eval_') and not any(x in k for x in NOT_METRICS)})
    key = None
    if metric_name:   # the metric the Trainer uses to pick the best model
        name = metric_name if metric_name.startswith('eval_') else f'eval_{metric_name}'
        key = name if name in keys else None
    for k in keys if key is None else []:   # else the key whose maximum equals the best metric
        vals = [h[k] for h in evals if k in h]
        if best_metric is not None and vals and abs(max(vals) - best_metric) < 1e-9:
            key = k
            break
    if key is None:
        named = [k for k in keys if any(x in k for x in ('acc', 'match', 'exact'))]
        key = named[0] if named else None
    if key is None:
        return None, None, None
    points = [(h.get('step', 0), h[key]) for h in evals if key in h]
    best = max(v for _, v in points)
    threshold = 0.99 if best <= 1.0 else 99.0
    hit = next((s for s, v in points if v >= threshold), None)
    return key, best, hit


def load_config(drive):
    p = drive / 'config.json'
    return json.load(open(p))['config'] if p.exists() else {}


def describe_settings(cfg):
    parts = [f"{cfg.get('num_kv_pairs')} pairs", f"K = {cfg.get('K')}", f"inner_lr {cfg.get('inner_lr')}",
             f"{cfg.get('n_layer')} layers", f"hidden {cfg.get('n_embd')}", f"seed {cfg.get('seed')}"]
    return ', '.join(p for p in parts if 'None' not in p)


def init_from(cfg):
    init = cfg.get('init_checkpoint') or cfg.get('init_base_checkpoint')
    if not init:
        return 'scratch'
    parts = Path(init).parts   # .../runs/<RUN>/<checkpoints/checkpoint-N | base>/model.safetensors
    if 'runs' in parts and parts.index('runs') + 2 < len(parts):
        i = parts.index('runs')
        return '/'.join(p for p in parts[i + 1:-1] if p != 'checkpoints')
    return init


class RunLog:
    """Keeps summary.json of one run up to date and writes its entries in runs.md."""

    def __init__(self, drive, note, commit, runs_md):
        self.drive, self.runs_md = drive, runs_md
        self.path = drive / 'summary.json'
        self.summary = json.load(open(self.path)) if self.path.exists() else {'run': drive.name, 'sessions': []}
        self.session = {'started': now(), 'commit': commit, 'note': note,
                        'start_step': None, 'end_step': None, 'seconds': 0.0}
        self.summary['sessions'].append(self.session)
        self.status, self.trainer, self.t0 = 'running', None, None

    def attach(self, trainer, start_step):
        self.trainer, self.t0 = trainer, time.time()
        self.session['start_step'] = start_step
        try:   # which device this session ran on, so speeds from different hardware are never mixed (fix 9)
            import torch
            self.session['device'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
        except Exception:
            self.session['device'] = 'unknown'

    def update(self):
        s = self.summary
        s['status'], s['last_update'] = self.status, now()
        if self.trainer is not None:
            st = self.trainer.state
            self.session['end_step'] = st.global_step
            self.session['seconds'] = round(time.time() - self.t0, 1)
            key, best, hit = eval_summary(list(st.log_history), st.best_metric,
                                          getattr(self.trainer.args, 'metric_for_best_model', None))
            s.update(steps=st.global_step, metric=key, best=best, step_to_99=hit)
        self.drive.mkdir(parents=True, exist_ok=True)
        write_json_atomic(self.path, s)

    def launch_entry(self, resumed_from):
        cfg = load_config(self.drive)
        start = f'Resumed from {resumed_from}' if resumed_from else 'New run from step 0'
        append_md(self.runs_md, '\n'.join([
            f"### {now()} · {self.drive.name} · launch",
            f"- {start}. Initial weights from {init_from(cfg)}.",
            f"- Settings. {describe_settings(cfg)}.",
            f"- Code {self.session['commit']}.",
            f"- Why. {self.session['note'] or 'no note'}"]))

    def end_entry(self):
        s = self.summary
        best = s.get('best')
        best_txt = f"best accuracy {best:.4f}" if best is not None else 'no eval yet'
        hit = s.get('step_to_99')
        hit_txt = f'99% reached at step {hit}' if hit is not None else '99% not reached'
        done = self.session['end_step'] is not None and self.session['start_step'] is not None
        n = (self.session['end_step'] - self.session['start_step']) if done else 0
        speed = f"{1000 * self.session['seconds'] / n:.0f} ms per step in this session" if n > 0 else 'no steps in this session'
        append_md(self.runs_md, '\n'.join([
            f"### {now()} · {self.drive.name} · {self.status}",
            f"- Step {s.get('steps', 0)}, {best_txt}, {hit_txt}.",
            f"- {speed}.",
            "- Conclusion. Add it with the conclusion cell."]))


RESULT_COLUMNS = ['run', 'status', 'pairs', 'K', 'inner_lr', 'layers', 'hidden', 'seed', 'init_from',
                  'steps', 'best_acc', 'step_to_99', 'ms_per_step', 'devices', 'sessions', 'code', 'last_update',
                  'why', 'conclusion']


def rebuild_results(runs_dir, csv_path):
    """Write results.csv from every run folder. Safe to call from several runs at once."""
    runs_dir, csv_path = Path(runs_dir), Path(csv_path)
    rows = []
    for d in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        cfg = load_config(d)
        s = json.load(open(d / 'summary.json')) if (d / 'summary.json').exists() else {}
        ses = s.get('sessions', [])
        done = [x for x in ses if x.get('start_step') is not None and x.get('end_step') is not None
                and x['end_step'] > x['start_step']]
        steps_done = sum(x['end_step'] - x['start_step'] for x in done)
        ms = round(1000 * sum(x['seconds'] for x in done) / steps_done, 1) if steps_done else None
        notes = [x['note'] for x in ses if x.get('note')]
        rows.append({
            'run': d.name, 'status': s.get('status', 'not started'),
            'pairs': cfg.get('num_kv_pairs'), 'K': cfg.get('K'), 'inner_lr': cfg.get('inner_lr'),
            'layers': cfg.get('n_layer'), 'hidden': cfg.get('n_embd'), 'seed': cfg.get('seed'),
            'init_from': init_from(cfg), 'steps': s.get('steps'), 'best_acc': s.get('best'),
            'step_to_99': s.get('step_to_99'), 'ms_per_step': ms,
            'devices': ' + '.join(sorted({x.get('device', 'unknown') for x in ses})) if ses else None,
            'sessions': len(ses),
            'code': ses[-1].get('commit') if ses else None, 'last_update': s.get('last_update'),
            'why': notes[-1] if notes else None, 'conclusion': s.get('conclusion')})
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = csv_path.with_name(csv_path.name + f'.{os.getpid()}.tmp')
    with open(tmp, 'w', newline='', encoding='utf8') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        w.writeheader()
        w.writerows(rows)
    replace_retry(tmp, csv_path)
    return rows


EVAL_KEYS = [('eval_loss', 'loss'), ('eval_target_loss', 'target'), ('eval_rec_loss', 'rec'),
             ('eval_inner_loss', 'inner'), ('eval_token_accuracy', 'tok_acc'),
             ('eval_all_queries_exact_match', 'exact')]


BAD_EVALS_TO_STOP = 3     # evals in a row with an eval loss above the limit
GUARD = {'reason': None}  # set when the divergence guard stopped the run


def make_metrics_log(callback_base, path, loss_limit):
    """A Trainer callback. Every logged value goes to metrics.jsonl (one JSON line per log event),
    and every evaluation prints one short console line with the main losses and accuracies.

    It is also the divergence guard. Training stops when a training loss is NaN or infinite, or when the
    eval target (read) loss is NaN, infinite or above loss_limit (2 ln(vocab size), far worse than random guessing) for
    BAD_EVALS_TO_STOP evaluations in a row. The run then ends with status 'diverged'."""

    class MetricsLog(callback_base):
        def __init__(self):
            self.last_train, self.mark, self.bad = {}, None, 0

        def stop(self, control, reason):
            GUARD['reason'] = reason
            control.should_training_stop = True
            print(f'\n[guard] {reason}. Training stopped, the run will be marked diverged.', flush=True)

        def on_log(self, args, state, control, logs=None, **kwargs):
            if not logs:
                return
            rec = {'step': state.global_step, 'time': round(time.time(), 1), **logs}
            try:
                with open(path, 'a', encoding='utf8') as f:
                    f.write(json.dumps(rec, default=float) + '\n')
            except OSError as e:
                print(f'[run] metrics.jsonl not written: {e}', flush=True)
            if GUARD['reason'] is None and state.global_step > 0:
                bad_train = [k for k, v in logs.items() if k.endswith('loss') and not k.startswith('eval_')
                             and isinstance(v, (int, float)) and not math.isfinite(v)]
                if bad_train:
                    self.stop(control, f'training {bad_train[0]} is {logs[bad_train[0]]} at step {state.global_step}')
                # the read (target) loss, so a reconstruction term never decides divergence (fairness fix 8)
                ev_loss = logs.get('eval_target_loss', logs.get('eval_loss'))
                if isinstance(ev_loss, (int, float)):
                    self.bad = self.bad + 1 if (not math.isfinite(ev_loss) or ev_loss > loss_limit) else 0
                    if self.bad >= BAD_EVALS_TO_STOP:
                        self.stop(control, f'eval target loss above {loss_limit:.1f} for {self.bad} evals in a row '
                                           f'(now {ev_loss:.4g} at step {state.global_step})')
            if not any(k.startswith('eval_') for k in logs):
                self.last_train = logs
                return control
            now, step = time.time(), state.global_step
            speed = ''
            if self.mark and step > self.mark[1]:
                speed = f' | {1000 * (now - self.mark[0]) / (step - self.mark[1]):.0f} ms/step'
            self.mark = (now, step)
            ev = ' '.join(f'{name} {logs[k]:.4f}' for k, name in EVAL_KEYS if k in logs)
            tr = ' '.join(f'{k} {self.last_train[k]:.4f}' for k in ('loss', 'target_loss', 'rec_loss', 'inner_loss')
                          if isinstance(self.last_train.get(k), (int, float)))
            print(chr(10) + f'[eval] step {step} | {ev}' + (f' | train {tr}' if tr else '') + speed, flush=True)
            return control

    return MetricsLog()


GRADMEM_SCRIPT, RMT_SCRIPT = 'run_gradmemgpt_on_mqar.py', 'run_rmt_on_mqar.py'
COMBINED_SCRIPT = 'run_enc_gradmem_on_mqar.py'
COMBINED_N_SCRIPT = 'run_enc_gradmem_n_on_mqar.py'   # combined_n (N1 write-step rule, C4 random start)
FUSED_SCRIPT = 'run_gradmem_fused_on_mqar.py'         # fused, fused_reference (forward reuse in the write step)


def check_run_type(run_type, script, script_args):
    """Sanity check before anything starts. Returns an error message, or None when the run is allowed.

    gradmem            GradMem must start from an RMT checkpoint (plan, D1 and D10).
    gradmem_reference  GradMem from scratch, only for the reference runs E2 and E3.
    rmt                RMT always with the reconstruction loss (plan, D11). A warm start must come from RMT.
    combined           encoder + GradMem (Idea 3). Both parts must start from an RMT checkpoint (plan, D1 to D3).
    combined_n         the same with run_enc_gradmem_n_on_mqar.py (N1 write-step rule, C4 random start).
    fused              GradMem with the fused write (run_gradmem_fused_on_mqar.py). Starts from an RMT checkpoint.
    fused_reference    the fused write from scratch (no init_checkpoint), the fair pair of gradmem_reference.
    The checkpoint's own config.json names the class that saved it, so it shows where the weights come from."""
    if not run_type:
        return None
    a = {script_args[i][2:]: script_args[i + 1] for i in range(len(script_args) - 1)
         if script_args[i].startswith('--')}
    init = a.get('init_checkpoint')

    def rmt_parent_problem():
        cfg_path = Path(init).parent / 'config.json'
        if not Path(init).is_file():
            return f'the starting checkpoint does not exist ({init})'
        if not cfg_path.is_file():
            return f'no config.json next to the starting checkpoint ({cfg_path})'
        c = json.load(open(cfg_path))
        if c.get('model_type') != 'rmt_2segm':
            return f'the starting checkpoint is not an RMT checkpoint (model_type {c.get("model_type")!r})'
        if not c.get('use_reconstruction_loss'):
            return 'the RMT parent was trained without the reconstruction loss (plan, D11)'
        base = c.get('base_config') or {}
        pairs = [('num_hidden_layers', 'n_layer'), ('hidden_size', 'n_embd'), ('vocab_size', 'vocab_size')]
        diff = [f'{arg} {base.get(key)} vs {a.get(arg)}' for key, arg in pairs
                if a.get(arg) is not None and str(base.get(key)) != str(a.get(arg))]
        if a.get('n_mem_tokens') is not None and str(c.get('n_mem_tokens')) != str(a.get('n_mem_tokens')):
            diff.append(f'n_mem_tokens {c.get("n_mem_tokens")} vs {a.get("n_mem_tokens")}')
        if diff:
            return 'the RMT parent has a different model size: ' + ', '.join(diff)
        return None

    if 'init_base_checkpoint' in a:
        return 'init_base_checkpoint is not allowed. It loads nothing from RMT checkpoints'
    if run_type == 'gradmem':
        if Path(script).name != GRADMEM_SCRIPT:
            return f'a gradmem run must use {GRADMEM_SCRIPT}, not {script}'
        if not init:
            return 'a gradmem run must start from RMT weights, but no init_checkpoint is given (plan, D1, D10)'
        return rmt_parent_problem()
    if run_type == 'fused':
        if Path(script).name != FUSED_SCRIPT:
            return f'a fused run must use {FUSED_SCRIPT}, not {script}'
        if not init:
            return 'a fused run must start from RMT weights, but no init_checkpoint is given (plan, D1)'
        return rmt_parent_problem()
    if run_type == 'fused_reference':
        if Path(script).name != FUSED_SCRIPT:
            return f'a fused_reference run must use {FUSED_SCRIPT}, not {script}'
        if init:
            return 'a fused_reference run trains from scratch, but init_checkpoint is given'
        return None
    if run_type in ('combined', 'combined_n'):
        expected = COMBINED_SCRIPT if run_type == 'combined' else COMBINED_N_SCRIPT
        if Path(script).name != expected:
            return f'a {run_type} run must use {expected}, not {script}'
        if not init:
            return 'a combined run must start from RMT weights, but no init_checkpoint is given (plan, D1)'
        return rmt_parent_problem()
    if run_type == 'gradmem_reference':
        if Path(script).name != GRADMEM_SCRIPT:
            return f'a gradmem_reference run must use {GRADMEM_SCRIPT}, not {script}'
        if init:
            return 'a gradmem_reference run trains from scratch, but init_checkpoint is given'
        return None
    if run_type == 'rmt':
        if Path(script).name != RMT_SCRIPT:
            return f'an rmt run must use {RMT_SCRIPT}, not {script}'
        if a.get('use_reconstruction_loss') != 'true':
            return 'an rmt run must use the reconstruction loss (plan, D11)'
        return rmt_parent_problem() if init else None
    return f'unknown run type {run_type!r}'


def main():
    argv = sys.argv[1:]
    split = argv.index('--')
    ap = argparse.ArgumentParser()
    ap.add_argument('--repo', required=True)
    ap.add_argument('--script', required=True)
    ap.add_argument('--local_dir', required=True)
    ap.add_argument('--drive_dir', required=True)
    ap.add_argument('--log', default='')
    ap.add_argument('--sync_minutes', type=float, default=10)
    ap.add_argument('--note', default='')
    ap.add_argument('--commit', default='')
    ap.add_argument('--runs_md', default='')
    ap.add_argument('--results_csv', default='')
    ap.add_argument('--run_type', default='', help='rmt, gradmem, gradmem_reference or combined. Empty skips the check')
    a = ap.parse_args(argv[:split])
    script_args = argv[split + 1:]

    problem = check_run_type(a.run_type, a.script, script_args)
    if problem:
        print(f'[run] REFUSED, {problem}. Nothing was started.', flush=True)
        sys.exit(2)
    if a.run_type:
        print(f'[run] run type {a.run_type} checked', flush=True)

    local, drive = Path(a.local_dir), Path(a.drive_dir)
    local.mkdir(parents=True, exist_ok=True)
    restore(local, drive)

    runlog = RunLog(drive, a.note, a.commit, a.runs_md)
    existing = complete_ckpts(local)
    runlog.launch_entry(existing[-1].name if existing else None)
    runlog.update()

    import transformers
    original_train = transformers.Trainer.train
    training_started = {'value': False}

    # Fail fast when a starting checkpoint leaves weights unloaded. The repo scripts load
    # init_checkpoint with strict=False and only log missing keys, so a wrong checkpoint or flag
    # would otherwise train from scratch without any error. Only loads before training are
    # checked; the Trainer's own resume loads are left alone.
    try:
        import torch
    except ImportError:
        torch = None
    if torch is not None:
        original_load_state_dict = torch.nn.Module.load_state_dict

        def checked_load_state_dict(self, state_dict, strict=True, assign=False):
            result = original_load_state_dict(self, state_dict, strict=strict, assign=assign)
            if not strict and not training_started['value']:
                missing = [k for k in result.missing_keys if not k.endswith('lm_head.weight')]  # tied to embeddings
                if missing:
                    raise RuntimeError(
                        f'[run] the starting checkpoint left {len(missing)} weights unloaded, '
                        f'for example {missing[:5]}. Check init_checkpoint, and never use '
                        f'init_base_checkpoint for an RMT parent.')
                print(f'[run] starting checkpoint loaded, no missing weights '
                      f'({len(result.unexpected_keys)} unused keys)', flush=True)
            return result

        torch.nn.Module.load_state_dict = checked_load_state_dict

    def train_with_resume(self, *args, **kwargs):
        training_started['value'] = True
        start_step = 0
        if kwargs.get('resume_from_checkpoint') is None:
            ck = complete_ckpts(Path(self.args.output_dir))
            if ck:
                kwargs['resume_from_checkpoint'] = str(ck[-1])
                start_step = step_of(ck[-1])
                print(f'[run] resuming from {ck[-1].name}', flush=True)
            else:
                print('[run] no checkpoint found, starting from step 0', flush=True)
        runlog.attach(self, start_step)
        if hasattr(transformers, 'TrainerCallback') and hasattr(self, 'add_callback'):
            self.add_callback(make_metrics_log(transformers.TrainerCallback,
                                               Path(self.args.output_dir) / 'metrics.jsonl', loss_limit))
        return original_train(self, *args, **kwargs)

    transformers.Trainer.train = train_with_resume
    args_map = {script_args[i][2:]: script_args[i + 1] for i in range(len(script_args) - 1)
                if script_args[i].startswith('--')}
    loss_limit = 2 * math.log(int(args_map.get('vocab_size', 8192)))   # twice the loss of random guessing

    # Fewer DataLoader workers on machines with few CPU cores (the repo scripts use 4).
    # The workers only collate batches, so the data and its order stay the same.
    workers = os.environ.get('GRADMEM_NUM_WORKERS')
    if workers is not None and hasattr(transformers, 'TrainingArguments'):
        original_post_init = transformers.TrainingArguments.__post_init__

        def post_init_with_workers(self):
            self.dataloader_num_workers = int(workers)
            original_post_init(self)

        transformers.TrainingArguments.__post_init__ = post_init_with_workers

    # The repo scripts create exp_path with mkdir(parents=True) and stop if it exists.
    # The run folder always exists here (restored checkpoints, resume), so allow it for this one folder.
    import pathlib
    original_mkdir = pathlib.Path.mkdir

    def mkdir_allow_run_dir(self, mode=0o777, parents=False, exist_ok=False):
        if self.resolve() == local.resolve():
            exist_ok = True
        return original_mkdir(self, mode=mode, parents=parents, exist_ok=exist_ok)

    pathlib.Path.mkdir = mkdir_allow_run_dir

    lock, stop = threading.Lock(), threading.Event()

    def safe_mirror(final=False):
        with lock:
            try:
                runlog.update()
                if a.results_csv:
                    rebuild_results(drive.parent, a.results_csv)
                mirror(local, drive, a.log, final=final)
            except Exception as e:  # a failed mirror must not stop training
                print(f'[run] mirror failed: {e}', flush=True)

    def loop():
        while not stop.wait(a.sync_minutes * 60):
            safe_mirror()

    threading.Thread(target=loop, daemon=True).start()
    signal.signal(signal.SIGTERM, lambda *_: sys.exit(143))   # so a stop still runs the final mirror

    os.chdir(a.repo)
    sys.path.insert(0, a.repo)
    sys.argv = [a.script] + script_args
    try:
        runpy.run_path(a.script, run_name='__main__')
        runlog.status = 'finished'
    except SystemExit as e:
        runlog.status = 'stopped' if e.code == 143 else ('finished' if e.code in (0, None) else f'failed (exit {e.code})')
        raise
    except BaseException as e:
        runlog.status = f'failed ({type(e).__name__})'
        raise
    finally:
        if GUARD['reason'] and runlog.status == 'finished':
            runlog.status = f"diverged ({GUARD['reason']})"
        stop.set()
        safe_mirror(final=True)
        try:
            runlog.end_entry()
        except Exception as e:
            print(f'[run] runs.md entry failed: {e}', flush=True)
        print(f'[run] {runlog.status}, final mirror to Drive done', flush=True)


if __name__ == '__main__':
    main()

## 5. Bring in runs from the inputs

Looks for run folders (a folder with the notebook's `config.json`) in every attached input, for example the dataset with runs downloaded from Drive, or the output of the previous version of this notebook. For each run name it takes the copy with the newest complete checkpoint and copies it into `/kaggle/working/gradmem_idea3/runs`. `runs.md` is brought in the same way.

Runs started on Colab store their starting checkpoint with a Drive path. The copy gets the same path under `/kaggle/working`, so the parent must be imported too (put it in the same dataset).

In [ ]:
import json, os, re, shutil, sys
from pathlib import Path
sys.path.insert(0, '/tmp')
import importlib, run_helper
importlib.reload(run_helper)

def find_runs(root, max_depth=10):
    # every folder under root that holds a run (a config.json with 'script' and 'config')
    found = []
    for d, sub, files in os.walk(root):
        rel = Path(d).relative_to(root).parts
        sub[:] = [s for s in sub if not s.endswith('.tmp') and not s.startswith('checkpoint-')
                  and len(rel) < max_depth]
        if 'config.json' in files:
            try:
                rec = json.load(open(Path(d) / 'config.json'))
            except (OSError, ValueError):
                continue
            if isinstance(rec, dict) and 'script' in rec and 'config' in rec:
                found.append(Path(d))
                sub[:] = []
    return found

def newness(d):
    ck = run_helper.complete_ckpts(d / 'checkpoints')
    step = run_helper.step_of(ck[-1]) if ck else -1
    try:
        last = json.load(open(d / 'summary.json')).get('last_update') or ''
    except (OSError, ValueError):
        last = ''
    return (step, last)

def make_writable(path):
    for d, _, files in os.walk(path):
        os.chmod(d, 0o755)
        for f in files:
            os.chmod(os.path.join(d, f), 0o644)

def fix_paths(run_dir):
    # a run started elsewhere (Colab) keeps its starting checkpoint under .../runs/<parent>/...
    p = run_dir / 'config.json'
    rec = json.load(open(p))
    init = rec['config'].get('init_checkpoint')
    if init and '/runs/' in init.replace('\\', '/'):
        new = f'{PROJ}/runs/' + init.replace('\\', '/').split('/runs/', 1)[1]
        if new != init:
            rec['config']['init_checkpoint'] = new
            json.dump(rec, open(p, 'w'), indent=2)
            print(f'  {run_dir.name}: starting checkpoint path set to {new}')

def import_runs():
    os.makedirs(f'{PROJ}/runs', exist_ok=True)
    os.makedirs(f'{PROJ}/results', exist_ok=True)
    if not os.path.isdir(INPUT_ROOT):
        print('No inputs attached.')
        return
    best = {}
    for d in find_runs(INPUT_ROOT):
        k = newness(d)
        if d.name not in best or k > best[d.name][0]:
            best[d.name] = (k, d)
    for name, (k, src) in sorted(best.items()):
        dst = Path(PROJ) / 'runs' / name
        if dst.exists() and newness(dst) >= k:
            continue
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns('*.tmp'))
        make_writable(dst)   # inputs are read-only, and copytree copies their permissions
        fix_paths(dst)
        print(f'Imported {name} (newest checkpoint {k[0] if k[0] >= 0 else "none"}) from {src}')
    md_files = [p for p in Path(INPUT_ROOT).rglob('runs.md') if p.parent.name == 'results']
    target = Path(PROJ) / 'results' / 'runs.md'
    if md_files:
        biggest = max(md_files, key=lambda p: p.stat().st_size)
        if not target.exists() or biggest.stat().st_size > target.stat().st_size:
            shutil.copyfile(biggest, target)
            print('Imported runs.md from', biggest)
    print('Runs available:', sorted(p.name for p in Path(PROJ, 'runs').iterdir() if p.is_dir()) or 'none')

import_runs()

## 6. Choose the runs

A queue of runs, trained in list order, two at a time. Each run has the same fields as cell 6 of the Colab notebook. A run never sees the other GPU, because with two visible GPUs the Trainer would split each batch over both and double the batch size.

| `RUN_TYPE` | What it is | Starts from |
|---|---|---|
| `'rmt'` | RMT, always with the reconstruction loss (D11). E1, E4, E5 | scratch, or an RMT parent (E4, E5 from E1) |
| `'gradmem'` | GradMem from RMT weights (D1, D10). E2r, E3r, E8r | **an RMT parent, required** |
| `'gradmem_reference'` | GradMem from scratch. Only E2 and E3 | scratch |
| `'fused'` | GradMem with the fused write, `run_gradmem_fused_on_mqar.py`. In `EXTRA`, `fused_mode` (`'sum'`, `'probe'`, `'none'`) and `fused_norm` (5.0) | **an RMT parent, required** |
| `'fused_reference'` | the same fused write from scratch | scratch, no parent |
| `'combined_n'` | The same model from `run_enc_gradmem_n_on_mqar.py`. In `EXTRA`, `inner_step_scale='mem_sq'` or `'normalized'` gives N1, `start_mode='random'` gives C4. With the defaults it equals `'combined'` | **an RMT parent, required** |
| `'combined'` | RMT encoder + GradMem (Idea 3). E6, E7. In `EXTRA`, `use_projection=True` gives E6p, `encoder_reconstruction_loss=True` gives E6rec and E7rec, `freeze_encoder=True` gives C1 | **an RMT parent, required** |

`K` means passes over the context for RMT and write steps for GradMem. `PARENT_RMT_RUN` is a finished RMT run that cell 5 imported. Without `PARENT_STEP` the parent's **best** checkpoint is used (the one in its `trainer_state.json`). A parent must already be finished when the session starts, because all configurations are built and checked before any training.

A run whose status is `finished` is skipped, so leaving it in the list wastes no compute. A run stopped by the divergence guard (status `diverged`, its loss became NaN or stayed far above random guessing) is skipped too. Add `FORCE=True` to its dict to train it further. A run that was stopped (time limit) or cut (disconnect) resumes.

A new run starts only while at least `MIN_HOURS_TO_START` remain before the time limit.

The learning rate stays constant for all models (D14).

`CONCLUSIONS` adds a conclusion to a finished run, for example `{'E2_gm_L4_p16_1M_s1': 'Reached 99% token accuracy at step 12k.'}`. It goes into `runs.md` and `results.csv` once.

In [ ]:
RUNS = [
    dict(RUN='E2_gm_L4_p16_1M_s1', NOTE='E2. GradMem K=1 reference from scratch',
         RUN_TYPE='gradmem_reference', K=1, PARENT_RMT_RUN=None, PARENT_STEP=None,
         EXTRA=dict()),
    dict(RUN='E3_gm_L4_p16_1M_s1', NOTE='E3. GradMem K=2 reference from scratch',
         RUN_TYPE='gradmem_reference', K=2, PARENT_RMT_RUN=None, PARENT_STEP=None,
         EXTRA=dict()),
]
ALLOW_SIZE_CHANGE = False   # True only on purpose, it breaks comparability with the other runs
TIME_LIMIT_HOURS = 11       # below Kaggle's 12 hours AND below your remaining weekly GPU quota
MIN_HOURS_TO_START = 0.5    # a queued run starts only if this much time is left
CONCLUSIONS = {}            # {'run name': 'one or two sentences'}
DRY_RUN = False             # True builds and checks every configuration, then stops before training

In [ ]:
import glob, json, os, re, time

# ---------- the shared base (plan, D11 to D14) ----------
BASE = dict(
    # data (MQAR), 16 pairs, 1M examples
    vocab_size=8192, num_kv_pairs=16, input_seq_len=48,
    query_sampling='uniform', power_a=0.01,
    train_num_examples=1000000, valid_num_examples=3000, data_seed=123,
    # model, 4 layers, hidden 128
    base_model='llama', n_layer=4, n_head=4, n_embd=128, attn_implementation='eager',
    n_mem_tokens=8,
    # training
    per_device_batch_size=64, gradient_accumulation_steps=1, total_batch_size=64,
    learning_rate=1e-4, max_steps=60000, warmup_steps=1000,   # one warmup for every run (fairness fix 4)
    eval_steps=500, logging_steps=500,
    early_stopping_patience=500, stop_on_metric_value=0.99,
    stop_on_metric_consecutive=3,   # 99 % exact match in 3 evals in a row ends a run (fairness fix 2)
    seed=143,
)
SIZE_KEYS = ['vocab_size', 'num_kv_pairs', 'input_seq_len', 'n_layer', 'n_head', 'n_embd',
             'n_mem_tokens', 'train_num_examples', 'data_seed']
MODEL_KEYS = ['vocab_size', 'n_layer', 'n_head', 'n_embd', 'n_mem_tokens', 'num_kv_pairs']

def parent_checkpoint(parent, step, cfg):
    # check the RMT parent and return the path of the checkpoint to start from
    pdir = f'{PROJ}/runs/{parent}'
    if not os.path.exists(f'{pdir}/config.json'):
        raise ValueError(f'Parent {parent} not found ({pdir}). Attach the input that holds it.')
    rec = json.load(open(f'{pdir}/config.json'))
    if rec.get('script') != 'run_rmt_on_mqar.py':
        raise ValueError(f'Parent {parent} is not an RMT run (script {rec.get("script")}).')
    pcfg = rec['config']
    if not pcfg.get('use_reconstruction_loss'):
        raise ValueError(f'Parent {parent} was trained without the reconstruction loss (plan, D11).')
    diff = {k: (pcfg.get(k), cfg.get(k)) for k in MODEL_KEYS if pcfg.get(k) != cfg.get(k)}
    if diff and not ALLOW_SIZE_CHANGE:
        raise ValueError(f'Parent {parent} has a different model size or pair count: {diff}')
    cks = {int(m.group(1)): p for p in glob.glob(f'{pdir}/checkpoints/checkpoint-*/model.safetensors')
           for m in [re.search(r'checkpoint-(\d+)', p)] if m}
    if not cks:
        raise ValueError(f'Parent {parent} has no checkpoint yet.')
    chosen = step
    if chosen is None:   # the best checkpoint, as recorded by the Trainer in the newest checkpoint
        chosen = max(cks)
        try:
            state = json.load(open(f'{pdir}/checkpoints/checkpoint-{chosen}/trainer_state.json'))
            m = re.search(r'checkpoint-(\d+)', state.get('best_model_checkpoint') or '')
            if m and int(m.group(1)) in cks:
                chosen = int(m.group(1))
        except (OSError, ValueError):
            pass
    if chosen not in cks:
        raise ValueError(f'Parent {parent} has no checkpoint-{chosen}. Available {sorted(cks)}')
    summary = f'{pdir}/summary.json'
    status = json.load(open(summary)).get('status') if os.path.exists(summary) else 'unknown'
    if status not in ('finished', 'stopped'):
        print(f'Note. Parent {parent} has status "{status}". Its best checkpoint may still change.')
    print(f'  parent {parent}, checkpoint-{chosen}')
    return cks[chosen]

def to_args(cfg):
    out = []
    for k, v in cfg.items():
        if v is None:
            continue
        out += [f'--{k}', ('true' if v else 'false') if isinstance(v, bool) else str(v)]
    return out

def build(spec):
    RUN, RUN_TYPE, K = spec['RUN'], spec['RUN_TYPE'], spec['K']
    PARENT_RMT_RUN, EXTRA = spec.get('PARENT_RMT_RUN'), dict(spec.get('EXTRA') or {})
    print(RUN)
    if RUN_TYPE == 'rmt':
        script = 'run_rmt_on_mqar.py'
        cfg = {**BASE, **dict(K=K, use_reconstruction_loss=True, reconstruction_loss_weight=1.0)}
    elif RUN_TYPE in ('gradmem', 'gradmem_reference'):
        script = 'run_gradmemgpt_on_mqar.py'
        cfg = {**BASE, **dict(memory_backend='prefix', K=K, last_K_second_order=K, inner_lr=0.4,
                              use_adam=False, grad_mode='second', freeze_backbone=False)}
    elif RUN_TYPE == 'combined':
        script = 'run_enc_gradmem_on_mqar.py'
        cfg = {**BASE, **dict(memory_backend='prefix', K=K, last_K_second_order=K, inner_lr=0.4, use_adam=False, grad_mode='second', freeze_backbone=False, use_projection=False, encoder_reconstruction_loss=False, encoder_reconstruction_loss_weight=1.0, freeze_encoder=False)}
    elif RUN_TYPE in ('fused', 'fused_reference'):
        script = 'run_gradmem_fused_on_mqar.py'
        cfg = {**BASE, **dict(memory_backend='prefix', K=K, last_K_second_order=K, inner_lr=0.4, use_adam=False, grad_mode='second', freeze_backbone=False, fused_mode='sum', fused_norm=5.0)}
    elif RUN_TYPE == 'combined_n':
        script = 'run_enc_gradmem_n_on_mqar.py'
        cfg = {**BASE, **dict(memory_backend='prefix', K=K, last_K_second_order=K, inner_lr=0.4, use_adam=False, grad_mode='second', freeze_backbone=False, use_projection=False, encoder_reconstruction_loss=False, encoder_reconstruction_loss_weight=1.0, freeze_encoder=False, inner_step_scale='none')}
    else:
        raise ValueError(f"{RUN}: RUN_TYPE must be 'rmt', 'gradmem', 'gradmem_reference', 'combined', 'combined_n', 'fused' or 'fused_reference'")

    bad = [k for k in EXTRA if k in SIZE_KEYS and EXTRA[k] != cfg.get(k)]
    if bad and not ALLOW_SIZE_CHANGE:
        raise ValueError(f'{RUN}: EXTRA changes the model size or data {bad}. Set ALLOW_SIZE_CHANGE = True only on purpose.')
    if 'init_base_checkpoint' in EXTRA:
        raise ValueError(f'{RUN}: init_base_checkpoint loads nothing for RMT checkpoints. Use PARENT_RMT_RUN.')
    if RUN_TYPE == 'rmt' and EXTRA.get('use_reconstruction_loss') is False:
        raise ValueError(f'{RUN}: RMT without the reconstruction loss stays at the copy baseline (plan, D11).')
    cfg.update(EXTRA)

    run_dir = f'{PROJ}/runs/{RUN}'
    saved = f'{run_dir}/config.json'
    old = json.load(open(saved)) if os.path.exists(saved) else None
    summary_path = f'{run_dir}/summary.json'
    status = json.load(open(summary_path)).get('status') if os.path.exists(summary_path) else None
    if status == 'finished' and not spec.get('FORCE'):
        print('  finished already, skipped (add FORCE=True to train it further)')
        return None
    if status and status.startswith('diverged') and not spec.get('FORCE'):
        print(f'  {status}. Skipped. Fix the setting and use a new RUN name, or add FORCE=True to retry')
        return None

    if RUN_TYPE in ('gradmem', 'combined', 'combined_n', 'fused') and not PARENT_RMT_RUN:
        raise ValueError(f"{RUN}: a '{RUN_TYPE}' run must start from RMT weights. Set PARENT_RMT_RUN (plan, D1, D10).")
    if RUN_TYPE in ('gradmem_reference', 'fused_reference') and PARENT_RMT_RUN:
        raise ValueError(f"{RUN}: 'gradmem_reference' and 'fused_reference' runs train from scratch. Use 'gradmem' or 'fused' to start from RMT.")
    if PARENT_RMT_RUN:
        if old is not None and 'init_checkpoint' in old['config']:
            cfg['init_checkpoint'] = old['config']['init_checkpoint']   # a resumed run keeps its original start
            print('  resumed run, keeping its original start', cfg['init_checkpoint'])
        else:
            cfg['init_checkpoint'] = parent_checkpoint(PARENT_RMT_RUN, spec.get('PARENT_STEP'), cfg)
    elif RUN_TYPE == 'gradmem_reference':
        print('  reference run from scratch (E2 or E3)')

    os.makedirs(run_dir, exist_ok=True)
    record = json.loads(json.dumps({'script': script, 'config': cfg}))
    if old is not None:
        changed = {k: (old['config'][k], v) for k, v in record['config'].items()
                   if k in old['config'] and old['config'][k] != v}
        if old.get('script') != record['script'] or changed:
            raise ValueError(f'{RUN} already exists with a different configuration {changed}. '
                             'Use a new RUN name, or restore the old configuration.')
        added = sorted(set(record['config']) - set(old['config']))
        if added:
            print('  settings added to the base after this run started are not applied to it:', added)
        cfg = dict(old['config'])   # a resumed run keeps its original configuration
        print('  same configuration as before, the run will resume')
    else:
        json.dump(record, open(saved, 'w'), indent=2)
        print('  new run, configuration saved')
    with open(f'{run_dir}/code_versions.txt', 'a') as f:
        f.write(f'{BRANCH} {COMMIT}\n')
    args = ['--exp_path', f'{LOCAL_RUNS}/{RUN}'] + to_args(cfg)
    return dict(spec, SCRIPT=script, ARGS=args, LOCAL_DIR=f'{LOCAL_RUNS}/{RUN}', RUN_DIR=run_dir)

names = [s['RUN'] for s in RUNS]
if len(set(names)) != len(names):
    raise ValueError('Two runs have the same name. They would train into the same folder.')
if N_GPUS < 2:
    print(f'Note. Only {N_GPUS} GPU, so runs train one at a time. Use GPU T4 x2 for two at once.')

PLANNED = [r for r in (build(s) for s in RUNS) if r is not None]
print('Queue:', [r['RUN'] for r in PLANNED] or 'empty')

# conclusions, written once per text
for run, text in CONCLUSIONS.items():
    p = Path(PROJ) / 'runs' / run / 'summary.json'
    if not text.strip():
        continue
    if not p.parent.exists():
        print('Conclusion skipped, no run named', run)
        continue
    summary = json.load(open(p)) if p.exists() else {'run': run, 'sessions': []}
    if summary.get('conclusion') == text:
        continue
    summary['conclusion'] = text
    run_helper.write_json_atomic(p, summary)
    run_helper.append_md(f'{PROJ}/results/runs.md',
                         f'### {time.strftime("%Y-%m-%d %H:%M")} · {run} · conclusion\n- {text}')
    print('Conclusion saved for', run)

## 7. Train the queue

Starts the first runs, one per GPU, and keeps the session alive. When a run ends, the next run in the queue takes its GPU. Every 10 minutes it prints the newest eval of each running run. At `TIME_LIMIT_HOURS` after the session start it stops the runs, each run saves its newest checkpoint into the output, and the next session continues from there.

Each run is started with `CUDA_VISIBLE_DEVICES` set to its own GPU and with mixed precision off, the same precision as on Colab.

In [ ]:
import json, os, re, signal, subprocess, time

def alive(pid):
    try:
        state = open(f'/proc/{pid}/stat').read().rsplit(')', 1)[1].split()[0]
    except FileNotFoundError:
        return False
    if state == 'Z':                      # finished, but not yet collected
        try:
            os.waitpid(pid, os.WNOHANG)
        except ChildProcessError:
            pass
        return False
    return True

def launch(r, gpu):
    pid_file = f'{LOGS}/{r["RUN"]}.pid'
    if os.path.exists(pid_file) and alive(int(open(pid_file).read())):
        print(f'{r["RUN"]} is still running in this session, not started twice')
        return None
    log = f'{LOGS}/{r["RUN"]}_{time.strftime("%Y%m%d-%H%M%S")}.log'
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),   # one GPU per run, so the batch never doubles
               ACCELERATE_MIXED_PRECISION='no')              # full precision, as on Colab (T4 has no bf16)
    cmd = ['python', '-u', HELPER, '--repo', REPO, '--script', r['SCRIPT'],
           '--local_dir', r['LOCAL_DIR'], '--drive_dir', r['RUN_DIR'],
           '--log', log, '--sync_minutes', str(SYNC_MINUTES),
           '--note', r['NOTE'], '--commit', f'{BRANCH} {COMMIT}', '--run_type', r['RUN_TYPE'],
           '--runs_md', f'{PROJ}/results/runs.md',
           '--results_csv', f'{PROJ}/results/results.csv', '--'] + r['ARGS']
    proc = subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    open(pid_file, 'w').write(str(proc.pid))
    print(f'{time.strftime("%H:%M")} started {r["RUN"]} on GPU {gpu}, pid {proc.pid}', flush=True)
    return (r['RUN'], proc, log)

def last_eval(log):
    # the newest 'eval step N: {...}' line of the log, shortened to the key metrics
    try:
        with open(log, 'rb') as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 300_000))
            text = f.read().decode('utf8', 'replace').replace('\r', '\n')
    except OSError:
        return 'no log yet'
    for line in reversed(text.splitlines()):
        m = re.search(r'eval step (\d+): (\{.*\})', line)
        if m:
            try:
                d = json.loads(m.group(2))
            except ValueError:
                continue
            keys = ['eval_loss', 'eval_token_accuracy', 'eval_all_queries_exact_match', 'eval_rec_loss']
            return f'step {m.group(1)}, ' + ', '.join(f'{k[5:]} {d[k]:.4f}' for k in keys if k in d)
    lines = [l for l in text.splitlines() if l.strip()]
    return 'no eval yet. ' + (lines[-1][-200:] if lines else '')

def stop_all(active):
    # SIGTERM to the main process of each run only, so it exits cleanly and makes its final save.
    # Its DataLoader workers end with it. Whatever is still alive after 10 minutes is killed.
    for name, p, _ in active.values():
        if p.poll() is None:
            os.kill(p.pid, signal.SIGTERM)
    end = time.time() + 600
    while any(p.poll() is None for _, p, _ in active.values()) and time.time() < end:
        time.sleep(5)
    for name, p, _ in active.values():
        if p.poll() is None:
            print(name, 'did not stop in 10 minutes, killed')
            os.killpg(p.pid, signal.SIGKILL)

os.makedirs(LOGS, exist_ok=True)
deadline = SESSION_START + TIME_LIMIT_HOURS * 3600
queue = [] if DRY_RUN else list(PLANNED)
active, ended = {}, []      # active maps a GPU to (name, process, log)
next_report = 0
if DRY_RUN:
    print('DRY_RUN is True. Every configuration was built and checked, nothing was trained.')
while queue or active:
    for gpu, (name, p, log) in list(active.items()):
        if p.poll() is not None:
            print(f'{time.strftime("%H:%M")} {name} ended, exit {p.returncode}. {last_eval(log)}', flush=True)
            ended.append((name, p.returncode, log))
            del active[gpu]
    if time.time() > deadline:
        print('Time limit reached, stopping the runs. Each run saves its newest checkpoint.', flush=True)
        stop_all(active)
        for name, p, log in active.values():
            ended.append((name, p.returncode, log))
        active = {}
        break
    left_h = (deadline - time.time()) / 3600
    for gpu in range(N_GPUS):
        if gpu not in active and queue:
            if left_h < MIN_HOURS_TO_START:
                print(f'{left_h:.1f} h left, not starting {[r["RUN"] for r in queue]}')
                queue = []
                break
            started = launch(queue.pop(0), gpu)
            if started:
                active[gpu] = started
    if time.time() >= next_report and active:
        print(f'--- {time.strftime("%H:%M")}, {(time.time() - SESSION_START) / 3600:.1f} h into the session', flush=True)
        for gpu, (name, p, log) in sorted(active.items()):
            print(f'GPU {gpu} {name}. {last_eval(log)}', flush=True)
        next_report = time.time() + 10 * 60
    if queue or active:
        time.sleep(30)

for name, code_, log in ended:
    print(f'\n{name}, exit {code_}. Last lines of the log')
    print(''.join(open(log, encoding='utf8', errors='replace').read().replace('\r', '\n').splitlines(True)[-6:]))

## 9. Results

Rebuilds `results/results.csv` from every run folder and shows it. The same table and `runs.md` are in the version's Output tab under `gradmem_idea3/results/`.

In [ ]:
import pandas as pd
run_helper.rebuild_results(f'{PROJ}/runs', f'{PROJ}/results/results.csv')
n_files = sum(len(f) for _, _, f in os.walk('/kaggle/working' if os.path.isdir('/kaggle/working') else PROJ))
size_gb = sum(os.path.getsize(os.path.join(d, x)) for d, _, f in os.walk(PROJ) for x in f) / 1e9
print(f'Output, {n_files} files, {size_gb:.2f} GB (Kaggle keeps up to 20 GB).')
if n_files > 400:
    print('Warning. Many output files. Kaggle may limit the number of output files. Remove finished runs '
          'that are no longer needed as parents from the inputs.')
pd.set_option('display.max_colwidth', 80)
pd.read_csv(f'{PROJ}/results/results.csv')